In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


bus_routes = {'2016': "MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop/MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop_Fall_2016.csv", 
              '2017': "MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop/MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop_Fall_2017.csv", 
              '2018': "MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop/MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop_Fall_2018.csv", 
              '2019': "MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop/MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop_Fall_2019.csv", 
              '2020': "MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop/MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop_Fall_2020.csv", 
              '2021': "MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop/MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop_Fall_2021.csv",
              '2022': "MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop/MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop_Fall_2022.csv", 
              '2023': "MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop/MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop_Fall_2023.csv",
              '2024': "MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop/MBTA_Bus_Ridership_by_Trip_Season_Route_Line_and_Stop_Fall_2024.csv"}


col = ["season","route_id","route_variant","direction_id","trip_start_time","day_type_id","day_type_name","stop_name","stop_id","stop_sequence","boardings","alightings","load","sample_size"
 ]


Ridership Questions 
- []  Look at how bus ridership has changed over time (pre vs post covid), although we see systemwide decreases are there routes with higher ridership or less significant decreases?
- []  Is data on overcrowding available, is it possible to get per-route levels of overcrowding, is it possible to get information over the course of day (rush hour vs. typical hours, week vs. weekend)?
- []  Can we compare the decrease in bus ridership to public transportation (T or commuter rail)? How does the relative decrease compare to overall decrease?
- []  What key bus routes are there (https://en.wikipedia.org/wiki/MBTA_key_bus_routes)? What percentage of overall ridership do they represent and are any underserved?

In [110]:
df = {}

for x in bus_routes:
    df[x] = pd.read_csv(bus_routes[x], low_memory=False )

pivot_dict = {}   

#x = year
for x in df:
    pivot_dict[x] = df[x].pivot_table(index=("trip_start_time", "route_id", "stop_name"), columns="day_type_name", values=("boardings", "alightings", "load"),aggfunc="mean"
)



Creating a hash table that counts intervals

In [111]:
from collections import defaultdict
import pandas as pd

hourly_counter = {}

for year, df_year in df.items():
    # 1) extract an integer hour 0–23
    if df_year['trip_start_time'].dtype == object:
        hours = (
            df_year['trip_start_time']
            .astype(str)
            .str.split(':')
            .str[0]
            .astype(int, errors='ignore')
        )
    else:
        hours = pd.to_numeric(df_year['trip_start_time'], errors='coerce').astype('Int64')

    df_year = df_year.assign(hour=hours).dropna(subset=['hour'])

    # 2) group by route, stop, and hour
    counts = (
        df_year
        .groupby(['route_id', 'stop_name', 'hour'])
        .size()
        .reset_index(name='count')
    )

    # 3) build nested dict: route → stop → (h,h+1) → count
    year_dict = defaultdict(lambda: defaultdict(dict))
    for _, row in counts.iterrows():
        route    = row['route_id']
        stop     = row['stop_name']
        h        = int(row['hour'])
        interval = (h, h + 1)
        year_dict[route][stop][interval] = int(row['count'])

    # convert defaultdicts to plain dicts
    hourly_counter[year] = {route: dict(stops) 
                             for route, stops in year_dict.items()}








In [112]:
# Example: look at 2024, route '15'
print("2023, route 15 hourly counts:")
for interval, cnt in sorted(hourly_counter['2023']['15'].items()):
    print(f"  {interval}: {cnt}")

2023, route 15 hourly counts:
  169 BOWDOIN ST OPP EUNICE ST: {(0, 1): 8, (3, 4): 6, (4, 5): 1, (5, 6): 5, (6, 7): 9, (7, 8): 7, (8, 9): 8, (9, 10): 9, (10, 11): 9, (11, 12): 8, (12, 13): 8, (13, 14): 9, (14, 15): 8, (15, 16): 7, (16, 17): 9, (17, 18): 7, (18, 19): 7, (19, 20): 10, (20, 21): 11, (21, 22): 9, (22, 23): 10, (23, 24): 9}
  452 GENEVA AVE OPP BLOOMFIELD: {(0, 1): 8, (3, 4): 3, (4, 5): 1, (5, 6): 5, (6, 7): 9, (7, 8): 7, (8, 9): 8, (9, 10): 9, (10, 11): 9, (11, 12): 8, (12, 13): 8, (13, 14): 9, (14, 15): 8, (15, 16): 7, (16, 17): 9, (17, 18): 7, (18, 19): 7, (19, 20): 10, (20, 21): 11, (21, 22): 9, (22, 23): 10, (23, 24): 9}
  ASHMONT BUSWAY: {(3, 4): 6}
  BOWDOIN OPP MT IDA: {(0, 1): 9, (1, 2): 4, (5, 6): 3, (6, 7): 7, (7, 8): 8, (8, 9): 7, (9, 10): 8, (10, 11): 9, (11, 12): 8, (12, 13): 8, (13, 14): 9, (14, 15): 9, (15, 16): 7, (16, 17): 9, (17, 18): 7, (18, 19): 8, (19, 20): 11, (20, 21): 11, (21, 22): 9, (22, 23): 9, (23, 24): 9}
  BOWDOIN ST @ ADAMS ST: {(0, 1): 8, (3,

We need to fine tune and find peak hours in 